# 2.0 — Variable factor weights (score-weighted)

In [ ]:
import sys, pathlib
SRC = pathlib.Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm

from paths import CLEAN_DATA, INTERIM_DATA
from metrics import total_return, ann_active_return
from chart import chart


# Load Data

In [ ]:
df_merge = pd.read_csv(CLEAN_DATA / '10y_merged.csv')
df_merge = df_merge.sort_values('date').set_index('date')


In [ ]:
df_factors = pd.read_csv(INTERIM_DATA / '10y_factors.csv')

factors = ['P/E', 'P/B', 'P/S', 'EV/EBITDA', 'FCF Yield', 'Earnings Yield']

df_factors[factors] = df_factors[factors].astype(float)
df_factors[factors] = stats.zscore(df_factors[factors])
zscore = df_factors.drop(columns=['12 Mo Yield', 'Company Id', 'SecId']).copy()
zscore[['P/E', 'P/B', 'P/S', 'EV/EBITDA']] = - zscore[['P/E', 'P/B', 'P/S', 'EV/EBITDA']]


# Train / Test Split

In [ ]:
_n_ = 50

split = int(_n_ / 100 * len(df_merge))

df_train = df_merge.iloc[:split]
df_test  = df_merge.iloc[split:]

bmk_train = df_train['sprtrn']
bmk_test  = df_test['sprtrn']

rf_train = df_train['rf']
rf_test  = df_test['rf']


# Variable factor weights — score-weighted

Random Dirichlet draws of factor weights; pick the configuration with best annualized active return on the train window, then evaluate on test.

In [ ]:
port_size = 30
trials_trian = 10**3


# Train

best_obj_train = -1e9
best_weight = None
best_tic = None
best_ret_train = None


df_m = df_train.copy()


for i in range(trials_trian):
    
    z_var = zscore.copy()
    
    onelist = np.ones(len(factors))
    weight = np.random.dirichlet(onelist)              # weighteight vector
    weight = weight / weight.sum()                     # making sure the sum is exactly 1             

    z_var[factors] = z_var[factors].mul(weight, axis=1) # multiply columns by weight
    z_var['score'] = z_var[factors].sum(axis=1)         # adding the weights
    
    z_var = z_var.sort_values('score', ascending=False).reset_index(drop=True)

    z_var_tic = z_var['Ticker'].iloc[:port_size]            # getting top tickers                        
    
    port_ret = df_m[z_var_tic]
    port_ret = port_ret.mean(axis=1)                     # weighteighted monthly avg return

    obj = ann_active_return(port_ret, bmk_train)           # objective function

    if obj > best_obj_train:
        best_obj_train = obj
        best_weight = weight.copy()
        best_tic = z_var_tic.copy()
        best_ret_train = port_ret.copy()
        
#########################################################################  
# Test

z_equal = zscore.copy()
df_m = df_test.copy()

# eqauly weighted z score
z_equal['score'] = z_equal[factors].mul(best_weight, axis=1).sum(axis=1)     
z_equal = z_equal.sort_values('score', ascending=False).reset_index(drop=True)   


# Portfolio weight optimization based on score
z_port = z_equal.iloc[:port_size].copy()
z_port['score weight'] = z_port['score'] / z_port['score'].sum()

z_port_weight = z_port['score weight'].astype(float).to_numpy()

port_tic = df_m[z_port['Ticker']].copy()
best_ret_test = port_tic.mul(z_port_weight, axis=1).sum(axis=1)

best_obj_test = ann_active_return(best_ret_test, bmk_test)

#########################################################################  
# final results

chart(best_ret_train, bmk_train, rf_train)
chart(best_ret_test, bmk_test, rf_test)

print("Best Active Return (Train):", round(best_obj_train, 2),'\n')
print("Best Active Return (Test):", round(best_obj_test, 2),'\n')

print('\nBest Factor Weights:')
for i, j in zip(factors, best_weight):
    print(f'{i}:', round(j * 100, 2))

z_port['score weight'] = round(z_port['score weight']*100 , 2)
display(z_port[['Ticker', 'score weight']])